# 01 - Probing (Phase 2/3), CPU only

Re-runs the corrected hidden-state probes on the canonical dataset.

* **Accelerator: None** (CPU). scikit-learn does not use the GPU; Kaggle's default 30 GB RAM is plenty.
* **Internet: On** (downloads 2.1 GB of hidden states from Hugging Face; public, no token needed).
* **Input:** the Kaggle Dataset made from `linguafranca_bundle.zip`.
* **Runtime:** ~5 min download + ~45-70 min for all 28 layers (set `LAYERS` below to e.g. `"8,9,10,11,12,13,14"` for ~15 min).

Outputs (`/kaggle/working/probing_results.zip`): `results.md` (tables), `results.json`, `test_predictions.csv`,
`qualitative_hop1.md`, and `probes/hop{1,2,12}_probe_layer{L}.joblib` (correctly trained, with metadata).

In [ ]:
# Find the uploaded bundle (any Kaggle Dataset containing lf_common.py) and work on a writable copy.
import os, sys, shutil
from pathlib import Path
hits = sorted(Path("/kaggle/input").rglob("lf_common.py"))
assert hits, ("Bundle not found. Click 'Add Input' (right panel) and add the Kaggle Dataset you created "
              "from linguafranca_bundle.zip.")
src = hits[0].parent
WORK = Path("/kaggle/working/lf")
if not WORK.exists():
    shutil.copytree(src, WORK)
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("bundle:", src, "->", WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Download the Phase-1 hidden states at the pinned revision (the one that matches the canonical dataset).
from huggingface_hub import snapshot_download
from lf_common import HF_REPO, HIDDEN_STATES_REVISION
snapshot_download(repo_id=HF_REPO, repo_type="dataset", revision=HIDDEN_STATES_REVISION,
                  allow_patterns=["data/hidden_states/*"], local_dir="/kaggle/working/hf", max_workers=16)
HS = "/kaggle/working/hf/data/hidden_states"
print(len(os.listdir(HS)), "hidden-state files (expected 4078)")

In [ ]:
# Optional provenance check: rebuild the canonical dataset from the pinned HF revision and compare checksums.
!python scripts/make_canonical_dataset.py --out /kaggle/working/rebuilt
import json, hashlib
bundled = hashlib.sha256(open("data/canonical/augmented_v2.jsonl", "rb").read()).hexdigest()
rebuilt = json.load(open("/kaggle/working/rebuilt/augmented_v2.stats.json"))["sha256"]
print("bundled == rebuilt:", bundled == rebuilt)

In [ ]:
LAYERS = "all"          # or e.g. "8,9,10,11,12,13,14" for a quick run
!python advanced_probing.py --data data/canonical/augmented_v2.jsonl --hs-dir {HS} --out outputs/probing --layers {LAYERS} --layer-chunk 28

In [ ]:
print(open("outputs/probing/results.md").read())

## Sanity checks
1. The new probes, scored on hop-1 **and** hop-2 rows (a real hop-1 probe is good on hop 1 and weak on hop 2).
2. The OLD Hugging Face probes (`probes/hop1_probe_layer*.joblib`): they were trained on hop 2 - look at the hop-1 FPR (0.5-0.9).

In [ ]:
!python evaluate_saved_probe.py --probes-dir outputs/probing/probes --hs-dir {HS} --layers 8,10,11,13
!python evaluate_saved_probe.py --legacy --hs-dir {HS} --layers 0,10,11

In [ ]:
# Split-stability of the hop-1 probe over 5 random grouped splits.
!python bootstrap_confident_wrong.py --hs-dir {HS} --layers 8,9,10,11,12,13,14

In [ ]:
# Package the results; download the zip from the Output section (right panel) of this notebook.
import shutil
shutil.make_archive("/kaggle/working/probing_results", "zip", "/kaggle/working/lf/outputs/probing")
print("Download /kaggle/working/probing_results.zip from the Output tab.")